# Qwen Hypergraph Method Comparison — полный автономный прогон

Самодостаточный ноутбук для **репрезентативного сравнения** методов сжатия экспертов
`Qwen/Qwen1.5-MoE-A2.7B-Chat`:

| Метод | Описание |
|---|---|
| `coact_pair_lszipit` | Baseline из статьи: co-activation pairs + LS-ZipIt |
| `direct_hypergraph_pca` | Прямой hypergraph grouping + PCA core |
| `pair_seeded_hypergraph_pca` | **Новый метод**: pair-seeded groups m∈{2,3,4} + PCA core |
| `pair_seeded_hypergraph_mean` | Negative control (mean core) |

**Протокол:**
1. Сбор артефактов (co-activation, pairs, experts, MLP inputs)
2. Single-layer downstream proxy на CommonsenseQA (best-first ranking)
3. Cumulative layer replacement до `delta_accuracy <= 1pp`

**Требования:** CUDA GPU (~24 GB VRAM для `RUN_MODE="full"`), PyTorch, transformers, datasets, pandas, matplotlib.

**Зависимости от файлов репозитория нет** — весь код внутри ноутбука.

Установите `RUN_MODE = "smoke"` для быстрой проверки на 4 слоях.


In [ ]:

# Optional: install dependencies if missing (uncomment on a fresh machine)
# %pip install -q torch transformers datasets matplotlib seaborn tqdm pandas numpy

import gc
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, logging as hf_logging

hf_logging.set_verbosity_error()

# --- Run mode ---
# "full" = representative paper-style comparison (24 layers, 1024 calib, 300 QA)
# "smoke" = quick sanity check on 4 layers
RUN_MODE = "full"

MODEL_NAME = "Qwen/Qwen1.5-MoE-A2.7B-Chat"
MODEL_DTYPE = torch.bfloat16
EXPERTS = 60
TOP_K = 4
SEED = 42
PAIRING_METHOD = "activation_correlation"
EVOLUTION_STEPS = 50
LOSSLESS_THRESHOLD = 0.01  # delta_accuracy <= 1pp

CALIB_TRAIN_DATASET = {"path": "cais/mmlu", "name": "all", "split": "auxiliary_train"}
CALIB_VAL_DATASET = {"path": "cais/mmlu", "name": "all", "split": "validation"}
QA_TASK = {"path": "tau/commonsense_qa", "split": "validation"}

if RUN_MODE == "smoke":
    LAYERS = list(range(4))
    TRAIN_SIZE = 128
    VALIDATION_SIZE = 128
    QA_NUM_EXAMPLES = 50
    QA_BATCH_SIZE = 4
    MAX_TRAIN_TOKENS_PER_GROUP = 128
    MAX_TEST_TOKENS_PER_GROUP = 64
    MIN_TOKENS_PER_GROUP = 8
else:
    LAYERS = list(range(24))
    TRAIN_SIZE = 1024
    VALIDATION_SIZE = 1024
    QA_NUM_EXAMPLES = 300
    QA_BATCH_SIZE = 8
    MAX_TRAIN_TOKENS_PER_GROUP = 512
    MAX_TEST_TOKENS_PER_GROUP = 256
    MIN_TOKENS_PER_GROUP = 8

DATA_DIR = "./qwen_hypergraph_run_data"
RESULTS_DIR = "./qwen_hypergraph_run_results"
SKIP_ARTIFACTS_IF_PRESENT = True

assert torch.cuda.is_available(), "This notebook requires a CUDA GPU"
device = torch.device("cuda")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Device:", device)
print("Run mode:", RUN_MODE)
print("Layers:", len(LAYERS))
print("Calibration:", CALIB_TRAIN_DATASET, "size", TRAIN_SIZE)
print("Downstream QA:", QA_TASK, "size", QA_NUM_EXAMPLES)


## Базовые утилиты MLP / ZipIt / pairing (из оригинального пайплайна)


In [ ]:
import torch
from torch import nn

default_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def unite_matrix(mat1: torch.Tensor, mat2: torch.Tensor) -> torch.Tensor:
    """Unite matrix A and B in block format:
    A 0
    0 B"""
    add1 = torch.zeros((mat1.shape[0], mat2.shape[1]), device=mat1.device)
    add2 = torch.zeros((mat2.shape[0], mat1.shape[1]), device=mat1.device)
    mat1 = torch.cat([mat1, add1], dim=1)
    mat2 = torch.cat([add2, mat2], dim=1)
    return torch.cat([mat1, mat2], dim=0)


def build_matches(corr_matrix):
    """Get a matrix of correlations and
    build matching with most sum of correlations (greedy)"""
    matches = []
    n = corr_matrix.shape[0]
    used = torch.zeros(n, dtype=torch.bool)

    sim_matrix = corr_matrix.clone()

    sim_matrix.fill_diagonal_(-float("inf"))

    for _ in range(n // 2):
        _, max_idx = torch.max(sim_matrix.view(-1), 0)
        i = max_idx.item() // n
        j = max_idx.item() % n

        matches.append((i, j))
        used[i] = True
        used[j] = True

        sim_matrix[i, :] = -float("inf")
        sim_matrix[:, i] = -float("inf")
        sim_matrix[j, :] = -float("inf")
        sim_matrix[:, j] = -float("inf")

    del sim_matrix, used
    return matches


class Zipper(nn.Linear):
    """Save all inputs and build M and U matrix
    based on greedy matching in correlation matrix"""

    def __init__(self, features):
        super().__init__(features, features)
        self.M = None
        self.U = None
        self.activations = []

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if len(x.shape) == 1:
            x = torch.stack([x])
        self.activations.append(x)
        return x

    def build_MU(self):
        """Count M and U matrixes, using saved while training activations"""
        assert len(self.activations) > 0
        self.dtype = self.activations[0].dtype
        activations = torch.cat(self.activations, dim=0)
        corr_matrix = torch.corrcoef(activations.T)
        n = corr_matrix.shape[0]

        matches = build_matches(corr_matrix)
        self.M = torch.zeros(n // 2, n).to(self.activations[0].device)
        for idx, (i, j) in enumerate(matches):
            self.M[idx, i] = 0.5
            self.M[idx, j] = 0.5
        self.U = 2 * self.M.T.to(dtype=self.activations[0].dtype)

        self.activations = []
        del activations


class MLP(nn.Module):
    """Base interface for all models"""

    def __init__(self):
        super().__init__()

    def forward_unzip(self, x: torch.Tensor, out=None):
        """Count forward and return features of model 1 and model 2 concated"""
        y = self.forward(x.to(dtype=self.gate.weight.dtype))
        if self.U is not None:
            y = y.to(dtype=self.U.dtype)
            if out is not None:
                return torch.nn.functional.linear(
                    y, torch.chunk(self.U, 2, dim=0)[out]
                ).to(dtype=x.dtype)
            return torch.nn.functional.linear(y, self.U).to(dtype=x.dtype)
        if out is not None:
            return y.to(dtype=x.dtype)
        return y.repeat(*([1] * (y.dim() - 1) + [2])).to(dtype=x.dtype)

    def forward(self, x: torch.Tensor):
        raise NotImplementedError()

    def zip_with(self, other: "MLP"):
        """Make an prezipped model with other"""
        raise NotImplementedError()

    def train_with(self, train_data):
        """Run train data and eval M and U matrix"""
        for example in train_data:
            self.forward(example.to(next(self.parameters()).device))
        self.zip1.build_MU()
        self.zip2.build_MU()

    def zip(self):
        """ZipIt operation"""
        raise NotImplementedError()


class SimpleMLP(MLP):
    """MLP with two hidden linear layers"""

    def __init__(
        self,
        in_features: int,
        hid_features: int,
        out_features: int,
        bias: bool = False,
        device=default_device,
    ):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hid_features, bias=bias, device=device)
        self.zip1 = Zipper(hid_features)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hid_features, out_features, bias=bias, device=device)
        self.zip2 = Zipper(out_features)
        self.zipped = False
        self.U = None
        self.to(device)

    def forward(self, x: torch.Tensor):
        if self.zipped:
            x = torch.cat([x, x], dim=-1)
            x = self.fc1(x)
            x = self.zip1(x)
            x = self.act(x)
            x = self.fc2(x)
            x = self.zip2(x)
        else:
            x = self.fc1(x)
            x = self.act(x)
            x = self.fc2(x)
        return x

    def zip_with(self, other: "SimpleMLP"):
        """Make an prezipped model with other"""
        self.fc1.weight = nn.Parameter(unite_matrix(self.fc1.weight, other.fc1.weight))
        self.fc2.weight = nn.Parameter(unite_matrix(self.fc2.weight, other.fc2.weight))
        self.zipped = True

    def zip(self):
        """ZipIt operation"""
        self.U = nn.Parameter(self.zip2.U)

        self.fc1.weight = nn.Parameter(
            self.zip1.M
            @ self.fc1.weight
            @ torch.cat(
                2 * [torch.eye(self.fc1.in_features, device=self.fc1.weight.device)]
            )
        )
        self.fc2.weight = nn.Parameter(self.zip2.M @ self.fc2.weight @ self.zip1.U)

        del self.zip1
        self.zip1 = None
        del self.zip2
        self.zip2 = None

        self.zipped = False


class AdvancedMLP(MLP):
    """MLP with up, down and proj linear layers"""

    def __init__(
        self,
        in_features: int,
        hid_features: int,
        out_features: int,
        bias: bool = False,
        device=default_device,
    ):
        super().__init__()
        self.in_features = in_features
        self.hid_features = hid_features
        self.out_features = out_features
        self.gate = nn.Linear(in_features, hid_features, bias=bias, device=device)
        self.act = nn.SiLU()
        self.up = nn.Linear(in_features, hid_features, bias=bias, device=device)
        self.zip1 = None
        self.down = nn.Linear(hid_features, out_features, bias=bias, device=device)
        self.zip2 = None
        self.zipped = False
        self.U = None
        self.to(device)

    def forward(self, x: torch.Tensor):
        dtype = x.dtype
        x = x.to(dtype=self.gate.weight.dtype)
        if self.zipped:
            x = torch.cat([x, x], dim=-1)
            gate = self.act(self.gate(x))
            up = self.up(x)
            x = torch.mul(gate, up)
            x = self.zip1(x)
            x = self.down(x)
            x = self.zip2(x)
        else:
            x = self.act(self.gate(x)) * self.up(x)
            if self.down is not None:
                x = self.down(x)
        return x.to(dtype=dtype)

    def zip_with(self, other: "MLP"):
        """Make an prezipped model with other"""
        self.zip1 = Zipper(self.hid_features)
        self.zip2 = Zipper(self.out_features)
        self.gate.weight = nn.Parameter(
            unite_matrix(self.gate.weight, other.gate.weight)
        )
        self.up.weight = nn.Parameter(unite_matrix(self.up.weight, other.up.weight))
        self.down.weight = nn.Parameter(
            unite_matrix(self.down.weight, other.down.weight)
        )
        self.zipped = True

    def zip(self):
        """ZipIt operation"""
        self.U = nn.Parameter(self.zip2.U.to(dtype=self.gate.weight.dtype))

        self.gate.weight = nn.Parameter(
            self.zip1.M
            @ self.gate.weight
            @ torch.cat(
                2 * [torch.eye(self.gate.in_features, device=self.gate.weight.device)]
            )
        )
        self.up.weight = nn.Parameter(
            self.zip1.M
            @ self.up.weight
            @ torch.cat(
                2 * [torch.eye(self.up.in_features, device=self.gate.weight.device)]
            )
        )
        self.down.weight = nn.Parameter(self.zip2.M @ self.down.weight @ self.zip1.U)

        del self.zip1
        self.zip1 = None
        del self.zip2
        self.zip2 = None

        self.zipped = False


In [ ]:
import copy
import time
import torch
from tqdm import tqdm
import torch.nn.functional as F
from torch import nn


default_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FUSION_METHODS = {}


def _as_tensor(data):
    if torch.is_tensor(data):
        return data
    return torch.stack(data, dim=0)


def _as_2d_tensor(data, device=None, dtype=None):
    tensor = _as_tensor(data)
    if tensor.dim() == 1:
        tensor = tensor.unsqueeze(0)
    if device is not None or dtype is not None:
        tensor = tensor.to(
            device=device if device is not None else tensor.device, dtype=dtype
        )
    return tensor


def _model_device(model: nn.Module) -> torch.device:
    return next(model.parameters()).device


def _hidden_activations(model: AdvancedMLP, x: torch.Tensor) -> torch.Tensor:
    x = x.to(device=_model_device(model), dtype=model.gate.weight.dtype)
    return model.act(model.gate(x)) * model.up(x)


def _channel_scores(model: AdvancedMLP, x: torch.Tensor) -> torch.Tensor:
    hidden = _hidden_activations(model, x)
    act_score = hidden.float().pow(2).mean(dim=0).sqrt()
    weight_score = (
        model.gate.weight.float().norm(dim=1)
        + model.up.weight.float().norm(dim=1)
        + model.down.weight.float().norm(dim=0)
    )
    return act_score + 0.05 * weight_score


def _rank_hidden_channels(model: AdvancedMLP, x: torch.Tensor) -> torch.Tensor:
    scores = _channel_scores(model, x)
    return torch.argsort(scores, descending=True)


def _reorder_advanced_mlp(
    model: AdvancedMLP, order: torch.Tensor, create_new: bool = True
) -> AdvancedMLP:
    new_mlp = model if not create_new else copy.deepcopy(model)
    with torch.no_grad():
        new_mlp.gate.weight = nn.Parameter(model.gate.weight[order].clone())
        new_mlp.up.weight = nn.Parameter(model.up.weight[order].clone())
        new_mlp.down.weight = nn.Parameter(model.down.weight[:, order].clone())
    return new_mlp


def count_mse(f, x, y):
    """Count MSE if f(x) and y"""
    return torch.mean((f(x) - y) ** 2)


def eval_merger(
    model1: MLP, model2: MLP, merged_model: MLP, data: torch.Tensor
) -> torch.Tensor:
    """Count MSE of zipped model with two simple models"""
    with torch.no_grad():
        return count_mse(
            merged_model.forward_unzip,
            data,
            torch.cat([model1.forward(data), model2.forward(data)], dim=-1),
        )


def zipMLP(
    mlpA: MLP, mlpB: MLP, train_data: torch.Tensor, create_new: bool = True
) -> MLP:
    """Take two MLP models, zipit using train data. Return zipped MLP"""
    if len(train_data) == 0:
        return avgMLP(mlpA, mlpB, train_data, create_new)
    with torch.no_grad():
        new_mlp = mlpA
        if create_new:
            new_mlp = copy.deepcopy(mlpA)
        new_mlp.zip_with(mlpB)
        new_mlp.train_with(train_data)
        new_mlp.zip()
    return new_mlp


def avgMLP(
    mlpA: MLP, mlpB: MLP, train_data: torch.Tensor, create_new: bool = True
) -> MLP:
    """Take two MLP models. Return same model with parameters averaged"""
    with torch.no_grad():
        new_mlp = mlpA
        if create_new:
            new_mlp = copy.deepcopy(mlpA)
        new_dict = dict()
        dictA = mlpA.state_dict()
        dictB = mlpB.state_dict()
        for key in dictA.keys():
            new_dict[key] = (dictA[key] + dictB[key]) / 2
        new_mlp.load_state_dict(new_dict)
    return new_mlp


def procrustesMLP(
    mlpA: AdvancedMLP,
    mlpB: AdvancedMLP,
    train_data: torch.Tensor,
    create_new: bool = True,
) -> AdvancedMLP:
    """Align hidden channels by rank of activation signatures, then average.

    This is a lightweight, architecture-safe substitute for a full orthogonal
    Procrustes alignment: with gated MLP experts we can only permute hidden
    channels without breaking the nonlinearity structure, so we align by
    importance rank instead of applying a dense rotation.
    """
    if len(train_data) == 0:
        return avgMLP(mlpA, mlpB, train_data, create_new)

    with torch.no_grad():
        data = _as_2d_tensor(train_data, device=_model_device(mlpA))
        order_a = _rank_hidden_channels(mlpA, data)
        order_b = _rank_hidden_channels(mlpB, data)

        aligned_a = _reorder_advanced_mlp(mlpA, order_a, create_new=True)
        aligned_b = _reorder_advanced_mlp(mlpB, order_b, create_new=True)
        return avgMLP(aligned_a, aligned_b, train_data, create_new=create_new)


def _batch_iter(x: torch.Tensor, batch_size: int):
    for start in range(0, len(x), batch_size):
        yield x[start : start + batch_size]


def _estimate_fisher_diag(
    model: AdvancedMLP,
    data: torch.Tensor,
    teacher_output: torch.Tensor,
    teacher_hidden: torch.Tensor,
    *,
    batch_size: int = 256,
    hidden_loss_weight: float = 0.25,
) -> dict[str, torch.Tensor]:
    params = {name: p for name, p in model.named_parameters() if p.requires_grad}
    fisher = {
        name: torch.zeros_like(p, dtype=torch.float32) for name, p in params.items()
    }

    model.zero_grad(set_to_none=True)
    was_training = model.training
    model.eval()

    with torch.enable_grad():
        for batch_x, batch_teacher_out, batch_teacher_hidden in zip(
            _batch_iter(data, batch_size),
            _batch_iter(teacher_output, batch_size),
            _batch_iter(teacher_hidden, batch_size),
        ):
            model.zero_grad(set_to_none=True)
            pred_out = model(batch_x)
            pred_hidden = _hidden_activations(model, batch_x)
            loss = F.mse_loss(
                pred_out, batch_teacher_out
            ) + hidden_loss_weight * F.mse_loss(pred_hidden, batch_teacher_hidden)
            grads = torch.autograd.grad(
                loss,
                tuple(params.values()),
                retain_graph=False,
                create_graph=False,
                allow_unused=True,
            )
            for (name, _), grad in zip(params.items(), grads):
                if grad is None:
                    continue
                fisher[name] += grad.detach().float().pow(2)

    if was_training:
        model.train()
    return fisher


def fisherMLP(
    mlpA: AdvancedMLP,
    mlpB: AdvancedMLP,
    train_data: torch.Tensor,
    create_new: bool = True,
) -> AdvancedMLP:
    """Merge experts with parameter-wise Fisher precision weighting.

    We approximate Fisher information by squared gradients of a self-supervised
    distillation objective against the pairwise teacher ensemble.
    """
    if len(train_data) == 0:
        return avgMLP(mlpA, mlpB, train_data, create_new)

    with torch.no_grad():
        data = _as_2d_tensor(train_data, device=_model_device(mlpA))
        teacher_output = 0.5 * (mlpA(data) + mlpB(data))
        teacher_hidden = 0.5 * (
            _hidden_activations(mlpA, data) + _hidden_activations(mlpB, data)
        )

    fisher_a = _estimate_fisher_diag(mlpA, data, teacher_output, teacher_hidden)
    fisher_b = _estimate_fisher_diag(mlpB, data, teacher_output, teacher_hidden)

    merged = mlpA if not create_new else copy.deepcopy(mlpA)
    merged_state = merged.state_dict()
    state_a = mlpA.state_dict()
    state_b = mlpB.state_dict()

    for key in merged_state.keys():
        imp_a = fisher_a.get(key)
        imp_b = fisher_b.get(key)
        if imp_a is None or imp_b is None:
            merged_state[key] = (state_a[key] + state_b[key]) / 2
            continue

        imp_a = imp_a.float()
        imp_b = imp_b.float()
        # Normalize each tensor locally to keep the weighting numerically stable.
        imp_a = imp_a / (imp_a.mean() + 1e-8)
        imp_b = imp_b / (imp_b.mean() + 1e-8)
        merged_state[key] = (
            ((imp_a * state_a[key].float()) + (imp_b * state_b[key].float()))
            / (imp_a + imp_b + 1e-8)
        ).to(dtype=state_a[key].dtype)

    merged.load_state_dict(merged_state, strict=True)
    return merged


def tiesMLP(
    mlpA: AdvancedMLP,
    mlpB: AdvancedMLP,
    train_data: torch.Tensor,
    create_new: bool = True,
    trim_ratio: float = 0.2,
) -> AdvancedMLP:
    """Sparse sign-consensus merge inspired by TIES.

    The original TIES formulation assumes task vectors from a shared base model.
    In this project we only have a pair of experts, so we adapt the idea to a
    pairwise merge: keep strong updates, resolve sign conflicts by magnitude,
    and trim small-magnitude entries.
    """
    if len(train_data) == 0:
        return avgMLP(mlpA, mlpB, train_data, create_new)

    with torch.no_grad():
        merged = mlpA if not create_new else copy.deepcopy(mlpA)
        state_a = mlpA.state_dict()
        state_b = mlpB.state_dict()
        merged_state = {}

        for key in state_a.keys():
            wa = state_a[key].float()
            wb = state_b[key].float()
            same_sign = torch.sign(wa) == torch.sign(wb)
            chosen = torch.where(
                same_sign, 0.5 * (wa + wb), torch.where(wa.abs() >= wb.abs(), wa, wb)
            )

            if trim_ratio > 0:
                threshold = torch.quantile(chosen.abs().flatten(), trim_ratio)
                chosen = torch.where(
                    chosen.abs() >= threshold, chosen, torch.zeros_like(chosen)
                )

            merged_state[key] = chosen.to(dtype=state_a[key].dtype)

        merged.load_state_dict(merged_state, strict=True)
        return merged


def distillMLP(
    mlpA: AdvancedMLP,
    mlpB: AdvancedMLP,
    train_data: torch.Tensor,
    create_new: bool = True,
) -> AdvancedMLP:
    """Train a student expert to mimic the pairwise teacher ensemble.

    This is the most expensive merge rule, but it is usually the strongest when
    the two experts are meaningfully different and you care about actual output
    preservation more than a closed-form weight merge.
    """
    if len(train_data) == 0:
        return avgMLP(mlpA, mlpB, train_data, create_new)

    device = _model_device(mlpA)
    data = _as_2d_tensor(train_data, device=device)
    batch_size = min(256, max(1, len(data)))
    student = avgMLP(mlpA, mlpB, train_data, create_new=create_new)

    with torch.no_grad():
        teacher_out = 0.5 * (mlpA(data) + mlpB(data))
        teacher_hidden = 0.5 * (
            _hidden_activations(mlpA, data) + _hidden_activations(mlpB, data)
        )

    student.train()
    optimizer = torch.optim.AdamW(student.parameters(), lr=3e-4, weight_decay=0.0)
    epochs = 4 if len(data) >= 512 else 8

    with torch.enable_grad():
        for _ in range(epochs):
            for batch_x, batch_teacher_out, batch_teacher_hidden in zip(
                _batch_iter(data, batch_size),
                _batch_iter(teacher_out, batch_size),
                _batch_iter(teacher_hidden, batch_size),
            ):
                optimizer.zero_grad(set_to_none=True)
                pred_out = student(batch_x)
                pred_hidden = _hidden_activations(student, batch_x)
                loss = F.mse_loss(pred_out, batch_teacher_out) + 0.25 * F.mse_loss(
                    pred_hidden, batch_teacher_hidden
                )
                loss.backward()
                optimizer.step()

    student.eval()
    return student


def proj(
    mlpA: MLP, mlpB: MLP, train_data: torch.Tensor, create_new: bool = True
) -> MLP:
    return copy.deepcopy(mlpA)


def solve_linear(X, Y, lambda_ridge=1e-3):
    d_in = X.shape[1]
    XtX = X.T @ X + lambda_ridge * torch.eye(d_in, device=X.device, dtype=X.dtype)
    rhs = X.T @ Y

    W = torch.linalg.solve(XtX, rhs).T
    return W


def finetune(method):
    def inner(mlpA: MLP, mlpB: MLP, train_data: torch.Tensor, create_new: bool = True):
        newMlp = method(mlpA, mlpB, train_data, create_new)
        if len(train_data) < 2 * mlpA.down.out_features:
            return newMlp
        newMlp.down = None
        all_tokens = _as_tensor(train_data)
        real_res = torch.cat(
            [mlpA.forward(all_tokens), mlpB.forward(all_tokens)], dim=-1
        )
        new_res = newMlp.forward(all_tokens)
        newMlp.U = torch.nn.Parameter(
            solve_linear(new_res.float(), real_res.float()).to(
                dtype=mlpA.down.weight.dtype
            )
        )
        return newMlp

    return inner


FUSION_METHODS["ZipIt"] = zipMLP
FUSION_METHODS["Average"] = avgMLP
FUSION_METHODS["Projection"] = proj
FUSION_METHODS["Procrustes"] = procrustesMLP
FUSION_METHODS["Fisher"] = fisherMLP
FUSION_METHODS["TIES"] = tiesMLP
FUSION_METHODS["Distillation"] = distillMLP

FUSION_METHODS["FineTune_ZipIt"] = finetune(zipMLP)
FUSION_METHODS["FineTune_Average"] = finetune(avgMLP)
FUSION_METHODS["FineTune_Projection"] = finetune(proj)
FUSION_METHODS["FineTune_Procrustes"] = finetune(procrustesMLP)
FUSION_METHODS["FineTune_Fisher"] = finetune(fisherMLP)
FUSION_METHODS["FineTune_TIES"] = finetune(tiesMLP)
FUSION_METHODS["FineTune_Distillation"] = finetune(distillMLP)


@torch.no_grad()
def find_mlp_mse_err(
    merger, n: int, hidden: int, out: int, examples: int = 10_000, device=default_device
) -> torch.Tensor:
    """For a given merger make two random models,
    merge them, train and test on random data.
    Return it's MSE with original models"""
    with torch.no_grad():
        mlp1 = AdvancedMLP(n, hidden, out, device=device)
        mlp2 = AdvancedMLP(n, hidden, out, device=device)

        x_train = torch.rand((10 * examples, n), device=device)
        x_test = torch.rand((10 * examples, n), device=device)

        mlp_merged = merger(mlp1, mlp2, x_train)

        res = eval_merger(mlp1, mlp2, mlp_merged, x_test)

    del mlp_merged, x_train, x_test, mlp1, mlp2
    torch.cuda.empty_cache()
    return res


def find_mlp_mse_err_repeat(
    merger, n: int, hidden: int, out: int, repeat: int = 1000
) -> torch.Tensor:
    """Repeat previous test [repeat] times. Return a tensor of results"""
    return (
        torch.stack(
            [
                find_mlp_mse_err(merger, n, hidden, out, examples=1000)
                for _ in tqdm(range(repeat))
            ],
            dim=0,
        )
        .detach()
        .cpu()
    )


@torch.no_grad()
def turn_in_mlp(mlp: torch.nn.Module, device=default_device) -> MLP:
    """Extract matrixs from loaded mlp layer
    and create AdvancedMLP with same weights"""
    new_mlp = AdvancedMLP(
        mlp.gate_proj.in_features,
        mlp.gate_proj.out_features,
        mlp.down_proj.out_features,
        device=device,
    )
    new_mlp.gate = mlp.gate_proj
    new_mlp.up = mlp.up_proj
    new_mlp.down = mlp.down_proj
    return new_mlp


def mse(x, y):
    return torch.square(x - y).mean().item()


def cosine(x, y):
    return torch.cosine_similarity(x, y).mean().item()


def mae(x, y):
    return torch.abs(x - y).mean().item()


def rmse(x, y):
    return torch.sqrt(torch.square(x - y).mean()).item()


def relative_mse(x, y):
    denom = torch.square(x - x.mean(dim=0, keepdim=True)).mean().clamp_min(1e-8)
    return (torch.square(x - y).mean() / denom).item()


def r2_score(x, y):
    ss_res = torch.square(x - y).sum()
    ss_tot = torch.square(x - x.mean(dim=0, keepdim=True)).sum().clamp_min(1e-8)
    return (1.0 - ss_res / ss_tot).item()


def pearson_corr(x, y):
    x = x.flatten().float()
    y = y.flatten().float()
    x = x - x.mean()
    y = y - y.mean()
    denom = torch.sqrt((x.square().sum() * y.square().sum()).clamp_min(1e-8))
    return (x.mul(y).sum() / denom).item()


def softmax_kl_divergence(logits_p, logits_q):
    p = F.log_softmax(logits_p, dim=-1)
    q = F.softmax(logits_q, dim=-1)
    return F.kl_div(p, q, reduction="batchmean").item()


def topk_agreement(logits_p, logits_q, k=1):
    k = max(1, min(k, logits_p.shape[-1], logits_q.shape[-1]))
    top_p = torch.topk(logits_p, k=k, dim=-1).indices
    top_q = torch.topk(logits_q, k=k, dim=-1).indices
    if k == 1:
        return (top_p.squeeze(-1) == top_q.squeeze(-1)).float().mean().item()
    overlap = []
    for a, b in zip(top_p, top_q):
        overlap.append(len(set(a.tolist()) & set(b.tolist())) / float(k))
    return float(sum(overlap) / max(1, len(overlap)))


def prediction_entropy(logits):
    probs = F.softmax(logits, dim=-1)
    return (-torch.sum(probs * torch.log(probs.clamp_min(1e-8)), dim=-1)).mean().item()


def center_gram(K):
    n = K.size(0)
    one = torch.ones((n, n), device=K.device) / n
    return K - one @ K - K @ one + one @ K @ one


def linear_cka(X, Y):
    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)

    K = X @ X.T
    L = Y @ Y.T

    Kc = center_gram(K)
    Lc = center_gram(L)

    hsic = (Kc * Lc).sum()

    norm_x = torch.sqrt((Kc * Kc).sum())
    norm_y = torch.sqrt((Lc * Lc).sum())

    return (hsic / (norm_x * norm_y + 1e-8)).mean().item()


def js_divergence_from_logits(logits_p, logits_q):
    """
    logits_p, logits_q: (N, M)
    """
    p = F.softmax(logits_p, dim=-1)
    q = F.softmax(logits_q, dim=-1)

    m = 0.5 * (p + q)

    js = 0.5 * (
        F.kl_div(m.log(), p, reduction="batchmean")
        + F.kl_div(m.log(), q, reduction="batchmean")
    )

    return js.mean().item()


@torch.no_grad()
def timer(mlp: MLP, test_data, iters=3) -> float:
    """Apply mlp on data and return required time"""
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()
    for _ in range(iters):
        mlp.forward_unzip(test_data)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_time = time.perf_counter()
    return end_time - start_time


@torch.no_grad()
def test_model_on_data(fused_mlp, mlp1, mlp2, test_data):
    all_tokens = _as_tensor(test_data)
    real_res = torch.cat(
        [mlp1.forward(all_tokens), mlp2.forward(all_tokens)], dim=-1
    ).float()
    new_res = fused_mlp.forward_unzip(all_tokens).float()
    return {
        "MSE": mse(real_res, new_res),
        "RMSE": rmse(real_res, new_res),
        "MAE": mae(real_res, new_res),
        "RelMSE": relative_mse(real_res, new_res),
        "Cosine": cosine(real_res, new_res),
        "Pearson": pearson_corr(real_res, new_res),
        "R2": r2_score(real_res, new_res),
        "CKA": linear_cka(real_res, new_res),
        "JS": js_divergence_from_logits(real_res, new_res),
        "KL": softmax_kl_divergence(real_res, new_res),
        "Top1": topk_agreement(real_res, new_res, k=1),
        "Top5": topk_agreement(real_res, new_res, k=min(5, real_res.shape[-1])),
        "EntropyGap": abs(prediction_entropy(real_res) - prediction_entropy(new_res)),
        "Time": 1000 * timer(fused_mlp, all_tokens),
    }


# True is greater is better
best_metrics = {
    "MSE": False,
    "RMSE": False,
    "MAE": False,
    "RelMSE": False,
    "Cosine": True,
    "Pearson": True,
    "R2": True,
    "JS": False,
    "KL": False,
    "Top1": True,
    "Top5": True,
    "EntropyGap": False,
    "CKA": True,
}


In [ ]:
import os
from typing import Optional
import torch
from tqdm import tqdm


def random_pairs(n_experts: int) -> list[tuple[int, int]]:
    perm = torch.randperm(n_experts).tolist()
    return [(perm[2 * i], perm[2 * i + 1]) for i in range(n_experts // 2)]


def eval_match(sim_matrix: torch.Tensor, matches, top_k: int = 8) -> float:
    win = 0.0
    for i, j in matches:
        win += float(sim_matrix[i, j])
    return win / (float(sim_matrix.sum()) / top_k)


def evolutionary_match(
    sim_matrix: torch.Tensor,
    evolution_steps: int,
    progress: bool = False,
) -> list[tuple[int, int]]:
    assert sim_matrix.dim() == 2 and sim_matrix.shape[0] == sim_matrix.shape[1]
    n = sim_matrix.shape[0]

    best_win = -float("inf")
    best_pairs: list[tuple[int, int]] = []

    iterator = range(evolution_steps)
    if progress:
        iterator = tqdm(iterator)

    with torch.no_grad():
        for _ in iterator:
            perm = torch.randperm(n).tolist()
            pairs = set((perm[2 * i], perm[2 * i + 1]) for i in range(n // 2))
            friend = [-1] * n
            for i, j in pairs:
                friend[i] = j
                friend[j] = i

            def swap(i, j):
                if (i, friend[i]) in pairs:
                    pairs.remove((i, friend[i]))
                else:
                    pairs.remove((friend[i], i))
                if (j, friend[j]) in pairs:
                    pairs.remove((j, friend[j]))
                else:
                    pairs.remove((friend[j], j))
                pairs.add((i, j))
                pairs.add((friend[i], friend[j]))
                fi, fj = friend[i], friend[j]
                friend[fi], friend[fj] = fj, fi
                friend[i], friend[j] = j, i

            def win_from_swap(i, j):
                return (
                    sim_matrix[i, j]
                    + sim_matrix[friend[i], friend[j]]
                    - sim_matrix[i, friend[i]]
                    - sim_matrix[j, friend[j]]
                ).item()

            while True:
                delta = 0.0
                bi, bj = -1, -1
                for i in range(n):
                    for j in range(i + 1, n):
                        d = win_from_swap(i, j)
                        if d > delta:
                            delta = d
                            bi, bj = i, j
                if delta < 1e-6:
                    break
                swap(bi, bj)

            win = eval_match(sim_matrix, pairs)
            if win > best_win:
                best_win = win
                best_pairs = list(pairs)

    return best_pairs


def compute_activation_correlation_matrix(layer: int, data_dir: str) -> torch.Tensor:
    path = f"{data_dir}/activations/train_matrix_{layer}.pt"
    matrix = torch.load(path).cpu().float()
    matrix = matrix / matrix.sum()
    return matrix


def compute_output_similarity_matrix(
    layer: int,
    data_dir: str,
    n_experts: int,
    device: torch.device,
    num_samples: int = 4096,
    batch_size: int = 512,
) -> torch.Tensor:
    experts_dir = f"{data_dir}/experts/{layer}"
    inputs_path = f"{data_dir}/mlp_inputs/train_{layer}.pt"
    if not os.path.isdir(experts_dir):
        raise FileNotFoundError("PAIRING_METHOD='output_similarity' requires step 3 ")
    if not os.path.isfile(inputs_path):
        raise FileNotFoundError("PAIRING_METHOD='output_similarity' requires step 4 ")

    raw_inputs = torch.load(inputs_path)
    if isinstance(raw_inputs, list):
        inputs = torch.stack(raw_inputs, dim=0)
    else:
        inputs = raw_inputs
    if inputs.dim() == 3:
        inputs = inputs.reshape(-1, inputs.shape[-1])
    if inputs.shape[0] > num_samples:
        idx = torch.randperm(inputs.shape[0])[:num_samples]
        inputs = inputs[idx]
    inputs = inputs.to(device)

    norm_outputs: list[torch.Tensor] = []
    for i in range(n_experts):
        expert = torch.load(f"{experts_dir}/expert_{i}.pt", map_location=device)
        expert.eval()
        outs = []
        with torch.no_grad():
            for start in range(0, inputs.shape[0], batch_size):
                chunk = inputs[start : start + batch_size]
                out = expert(chunk).float()
                out = out / out.norm(dim=-1, keepdim=True).clamp_min(1e-8)
                outs.append(out)
        norm_outputs.append(torch.cat(outs, dim=0).cpu())
        del expert
        if device.type == "cuda":
            torch.cuda.empty_cache()

    stacked = torch.stack(norm_outputs, dim=0)  # [E, N, H]
    n = stacked.shape[1]
    sim = torch.einsum("inh,jnh->ij", stacked, stacked) / n
    return sim


def get_pairs(
    method: str,
    layer: int,
    n_experts: int,
    data_dir: str,
    device: Optional[torch.device] = None,
    evolution_steps: int = 0,
    progress: bool = False,
) -> list[tuple[int, int]]:
    if method == "random":
        return random_pairs(n_experts)

    if method == "activation_correlation":
        sim = compute_activation_correlation_matrix(layer, data_dir)
    elif method == "output_similarity":
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        sim = compute_output_similarity_matrix(layer, data_dir, n_experts, device)
    else:
        raise ValueError(
            f"Unknown PAIRING_METHOD: {method!r}. "
            f"Expected: 'activation_correlation', 'random', 'output_similarity'."
        )

    if evolution_steps > 0:
        return evolutionary_match(sim, evolution_steps, progress=progress)
    return build_matches(sim)


## Hypergraph runtime + QA evaluation


In [ ]:

# --- Hypergraph grouping, runtime fusion, QA evaluation ---

from collections import Counter
import csv
import math
import random
import warnings


METHOD_CONFIGS = {
    "coact_pair_lszipit": {"core": "zipit_ls", "alpha": 1.0, "lambda": 1e-3},
    "direct_hypergraph_pca": {"core": "pca", "alpha": 0.5, "lambda": 1e-4},
    "pair_seeded_hypergraph_pca": {"core": "pca", "alpha": 0.75, "lambda": 1e-4},
    "pair_seeded_hypergraph_mean": {"core": "mean", "alpha": 0.75, "lambda": 1e-4},
}

COMPARISON_METHODS = [
    "coact_pair_lszipit",
    "direct_hypergraph_pca",
    "pair_seeded_hypergraph_pca",
    "pair_seeded_hypergraph_mean",
]

QA_TASK = {
    "path": "tau/commonsense_qa",
    "split": "validation",
}


def normalize_coactivation(matrix: torch.Tensor) -> torch.Tensor:
    matrix = matrix.float().clone()
    matrix.fill_diagonal_(0)
    return matrix / matrix.max().clamp_min(1e-8)


def expert_loads(raw_matrix: torch.Tensor) -> torch.Tensor:
    diag = raw_matrix.float().diag().clamp_min(0)
    if float(diag.sum()) == 0:
        return torch.ones_like(diag) / max(1, diag.numel())
    return diag / diag.sum()


def group_score(group, coact, loads, lambda_load=0.0):
    pairs = [(i, j) for pos, i in enumerate(group) for j in group[pos + 1 :]]
    co = sum(float(coact[i, j]) for i, j in pairs) / max(1, len(pairs))
    load_var = float(torch.tensor([loads[i] for i in group]).var(unbiased=False))
    return co - lambda_load * load_var


def greedy_pairs(coact: torch.Tensor):
    n = coact.shape[0]
    available = set(range(n))
    pairs = []
    edges = [(float(coact[i, j]), i, j) for i in range(n) for j in range(i + 1, n)]
    for _, i, j in sorted(edges, reverse=True):
        if i in available and j in available:
            pairs.append((i, j))
            available.remove(i)
            available.remove(j)
    return pairs


def random_pairs(n_experts: int, seed: int):
    rng = random.Random(seed)
    perm = list(range(n_experts))
    rng.shuffle(perm)
    return [(perm[i], perm[i + 1]) for i in range(0, len(perm) - 1, 2)]


def validate_groups(groups, num_experts: int):
    all_experts = [int(x) for group in groups for x in group]
    counts = Counter(all_experts)
    unique_experts = set(all_experts)
    duplicate_experts = sorted([x for x, c in counts.items() if c > 1])
    uncovered_experts = sorted(set(range(num_experts)) - unique_experts)
    group_sizes = [len(g) for g in groups]
    return {
        "num_experts": num_experts,
        "num_groups": len(groups),
        "group_sizes": group_sizes,
        "num_m2": group_sizes.count(2),
        "num_m3": group_sizes.count(3),
        "num_m4": group_sizes.count(4),
        "unique_covered_experts": len(unique_experts),
        "overlap_count": len(all_experts) - len(unique_experts),
        "duplicate_experts": duplicate_experts,
        "uncovered_experts": uncovered_experts,
        "uncovered_count": len(uncovered_experts),
        "is_valid": len(duplicate_experts) == 0 and len(all_experts) == len(unique_experts),
    }


def direct_hypergraph_groups(coact, loads, *, group_sizes, neighbor_k=8, lambda_load=0.1):
    n = coact.shape[0]
    candidates = set()
    for i in range(n):
        neigh = torch.topk(coact[i], k=min(neighbor_k, n)).indices.tolist()
        neigh = [j for j in neigh if j != i]
        pool = [i] + neigh
        for size in group_sizes:
            if size < 2 or size > len(pool):
                continue
            if size == 2:
                for j in neigh:
                    candidates.add(tuple(sorted((i, j))))
            elif size == 3:
                for a_pos, a in enumerate(neigh):
                    for b in neigh[a_pos + 1 :]:
                        candidates.add(tuple(sorted((i, a, b))))
            elif size == 4:
                for a_pos, a in enumerate(neigh):
                    for b_pos, b in enumerate(neigh[a_pos + 1 :], start=a_pos + 1):
                        for c in neigh[b_pos + 1 :]:
                            candidates.add(tuple(sorted((i, a, b, c))))
    ranked = sorted(
        candidates,
        key=lambda g: (group_score(g, coact, loads, lambda_load), len(g)),
        reverse=True,
    )
    used = set()
    groups = []
    for group in ranked:
        if not any(i in used for i in group):
            groups.append(group)
            used.update(group)
    return groups


def pair_seeded_groups(coact, *, tau_group=0.3):
    pairs = greedy_pairs(coact)
    used_pairs = set()
    used_experts = set()
    groups = []
    pair_edges = []
    for a_idx, (i, j) in enumerate(pairs):
        for b_idx, (k, l) in enumerate(pairs[a_idx + 1 :], start=a_idx + 1):
            between = float((coact[i, k] + coact[i, l] + coact[j, k] + coact[j, l]) / 4)
            inside = float((coact[i, j] + coact[k, l]) / 2)
            pair_edges.append((0.5 * inside + 0.5 * between, a_idx, b_idx))
    for score, a_idx, b_idx in sorted(pair_edges, reverse=True):
        if score < tau_group:
            continue
        if a_idx in used_pairs or b_idx in used_pairs:
            continue
        candidate = tuple(sorted((*pairs[a_idx], *pairs[b_idx])))
        if any(expert in used_experts for expert in candidate):
            continue
        groups.append(candidate)
        used_pairs.add(a_idx)
        used_pairs.add(b_idx)
        used_experts.update(candidate)
    for idx, pair in enumerate(pairs):
        candidate = tuple(sorted(pair))
        if idx in used_pairs:
            continue
        if any(expert in used_experts for expert in candidate):
            continue
        groups.append(candidate)
        used_pairs.add(idx)
        used_experts.update(candidate)
    return groups


def build_groups(method, coact, loads, num_experts, seed=0):
    if method == "coact_pair_lszipit":
        return greedy_pairs(coact)
    if method == "direct_hypergraph_pca":
        return direct_hypergraph_groups(coact, loads, group_sizes=[2, 3, 4], neighbor_k=8, lambda_load=0.1)
    if method in {"pair_seeded_hypergraph_pca", "pair_seeded_hypergraph_mean"}:
        return pair_seeded_groups(coact, tau_group=0.3)
    raise ValueError(method)


def selected_indices(activations, group, max_tokens):
    wanted = set(group)
    indices = []
    for idx, selected in enumerate(activations):
        if any(int(x) in wanted for x in selected.flatten()):
            indices.append(idx)
            if max_tokens is not None and len(indices) >= max_tokens:
                break
    return indices


def hidden_activations(expert, x):
    x = x.to(dtype=expert.gate.weight.dtype)
    return expert.act(expert.gate(x)) * expert.up(x)


def solve_ridge(h, y, ridge_lambda):
    h = h.float()
    y = y.float()
    eye = torch.eye(h.shape[1], device=h.device, dtype=h.dtype)
    a = h.T @ h + ridge_lambda * eye
    b = h.T @ y
    return torch.linalg.solve(a, b)


def center_gram(gram):
    n = gram.size(0)
    unit = torch.ones((n, n), device=gram.device) / n
    return gram - unit @ gram - gram @ unit + unit @ gram @ unit


def linear_cka(x, y):
    x = x - x.mean(dim=0, keepdim=True)
    y = y - y.mean(dim=0, keepdim=True)
    k = center_gram(x @ x.T)
    l = center_gram(y @ y.T)
    hsic = (k * l).sum()
    return float((hsic / (torch.sqrt((k * k).sum()) * torch.sqrt((l * l).sum()) + 1e-8)).item())


def js_divergence_from_logits(x, y):
    p = F.softmax(x, dim=-1)
    q = F.softmax(y, dim=-1)
    m = 0.5 * (p + q)
    js = 0.5 * (F.kl_div(m.log(), p, reduction="batchmean") + F.kl_div(m.log(), q, reduction="batchmean"))
    return float(js.item())


def reconstruction_metrics(y_true, y_pred):
    y_true = y_true.float()
    y_pred = y_pred.float()
    flat_true = y_true.reshape(y_true.shape[0], -1)
    flat_pred = y_pred.reshape(y_pred.shape[0], -1)
    sample = min(512, flat_true.shape[0])
    return {
        "MSE": float((flat_pred - flat_true).square().mean()),
        "normalized_MSE": float((flat_pred - flat_true).square().sum() / flat_true.square().sum().clamp_min(1e-8)),
        "Cosine": float(F.cosine_similarity(flat_pred, flat_true, dim=-1).mean()),
        "CKA": linear_cka(flat_true[:sample], flat_pred[:sample]),
        "JS": js_divergence_from_logits(flat_true, flat_pred),
    }


def layer_parameter_reduction(groups, *, num_experts, d_model, d_ff, r_by_group):
    validity = validate_groups(groups, num_experts)
    original_layer = num_experts * 3 * d_model * d_ff
    new_layer = 0
    for group_id, group in enumerate(groups):
        r = r_by_group.get(group_id, d_ff)
        new_layer += 2 * d_model * r + len(group) * r * d_model
    new_layer += validity["uncovered_count"] * 3 * d_model * d_ff
    return {
        **validity,
        "P_orig_layer": int(original_layer),
        "P_new_layer": int(new_layer),
        "CR_actual": float(1.0 - new_layer / max(1, original_layer)),
    }


class HypergraphGroup(nn.Module):
    def __init__(self, experts, *, core, basis, mean, w, group):
        super().__init__()
        self.experts = nn.ModuleList(experts)
        self.core = core
        self.register_buffer("basis", basis if basis is not None else torch.empty(0), persistent=True)
        self.register_buffer("mean", mean if mean is not None else torch.empty(0), persistent=True)
        self.register_buffer("w", w, persistent=True)
        self.group = tuple(group)

    def forward_side(self, x, side):
        h_cat = torch.cat([hidden_activations(expert, x) for expert in self.experts], dim=-1).float()
        if self.core == "mean":
            d_ff = h_cat.shape[-1] // len(self.experts)
            h = h_cat.reshape(h_cat.shape[0], len(self.experts), d_ff).mean(dim=1)
            if h.shape[1] > self.w.shape[0]:
                h = h[:, : self.w.shape[0]]
            elif h.shape[1] < self.w.shape[0]:
                h = torch.cat([h, torch.zeros(h.shape[0], self.w.shape[0] - h.shape[1], device=h.device)], dim=1)
        else:
            h = (h_cat - self.mean) @ self.basis
        d_model = self.w.shape[1] // len(self.experts)
        return (h @ self.w[:, side * d_model : (side + 1) * d_model]).to(dtype=x.dtype)


class RuntimeFusionLayer(nn.Module):
    def __init__(self, *, num_experts, groups, group_modules, original_experts, gate, shared_expert, shared_expert_gate):
        super().__init__()
        self.num_experts = num_experts
        self.groups = [tuple(g) for g in groups]
        self.group_modules = nn.ModuleList(group_modules)
        self.original_experts = nn.ModuleList(original_experts)
        self.gate = gate
        self.shared_expert = shared_expert
        self.shared_expert_gate = shared_expert_gate
        self.expert_to_group_side = {}
        for group_id, group in enumerate(self.groups):
            for side, expert in enumerate(group):
                self.expert_to_group_side[int(expert)] = (group_id, side)

    def forward(self, hidden_states):
        batch_size, sequence_length, hidden_dim = hidden_states.shape
        flat = hidden_states.view(-1, hidden_dim)
        shared = self.shared_expert(flat)
        _, routing_weights, selected_experts = self.gate(flat)
        expert_output = self._forward_experts(flat, selected_experts, routing_weights)
        shared = F.sigmoid(self.shared_expert_gate(flat)) * shared
        return (expert_output + shared).reshape(batch_size, sequence_length, hidden_dim)

    def _forward_experts(self, flat, top_k_index, top_k_weights):
        final = torch.zeros_like(flat)
        with torch.no_grad():
            expert_mask = F.one_hot(top_k_index, num_classes=self.num_experts).permute(2, 1, 0)
            expert_hit = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()
        for expert_idx_tensor in expert_hit:
            expert_idx = int(expert_idx_tensor[0])
            top_k_pos, token_idx = torch.where(expert_mask[expert_idx])
            current = flat[token_idx]
            if expert_idx in self.expert_to_group_side:
                group_id, side = self.expert_to_group_side[expert_idx]
                module = self.group_modules[group_id]
                if isinstance(module, HypergraphGroup):
                    out = module.forward_side(current, side)
                else:
                    group = self.groups[group_id]
                    out = module.forward_unzip(current, out=(0 if expert_idx == group[0] else 1))
            else:
                out = self.original_experts[expert_idx](current)
            out = out * top_k_weights[token_idx, top_k_pos, None]
            final.index_add_(0, token_idx, out.to(final.dtype))
        return final


def build_pca_core(h_train, r):
    train = h_train.float()
    mean = train.mean(dim=0, keepdim=True)
    centered = train - mean
    r_eff = max(1, min(r, centered.shape[0] - 1, centered.shape[1]))
    _, _, vh = torch.linalg.svd(centered, full_matrices=False)
    basis = vh[:r_eff].T.contiguous()
    h = centered @ basis
    return h, basis, mean, r_eff


def as_2d_tensor(data, device=None, dtype=None):
    if isinstance(data, list):
        tensor = torch.stack(data, dim=0)
    else:
        tensor = data
    if tensor.dim() == 3:
        tensor = tensor.reshape(-1, tensor.shape[-1])
    if device is not None or dtype is not None:
        tensor = tensor.to(device=device if device is not None else tensor.device, dtype=dtype)
    return tensor


def build_runtime_layer(layer, method, device):
    cfg = METHOD_CONFIGS[method]
    raw_matrix = torch.load(f"{DATA_DIR}/activations/train_matrix_{layer}.pt", weights_only=False).cpu()
    coact = normalize_coactivation(raw_matrix)
    loads = expert_loads(raw_matrix)
    groups = build_groups(method, coact, loads, EXPERTS, seed=SEED)
    validity = validate_groups(groups, EXPERTS)
    if not validity["is_valid"]:
        raise RuntimeError(f"invalid groups for {method} layer {layer}: {validity}")

    train_inputs_raw = torch.load(f"{DATA_DIR}/mlp_inputs/train_{layer}.pt", weights_only=False)
    test_inputs_raw = torch.load(f"{DATA_DIR}/mlp_inputs/test_{layer}.pt", weights_only=False)
    train_acts = torch.load(f"{DATA_DIR}/activations/train_all_{layer}.pt", weights_only=False)
    test_acts = torch.load(f"{DATA_DIR}/activations/test_all_{layer}.pt", weights_only=False)
    train_all = as_2d_tensor(train_inputs_raw, device=device)
    test_all = as_2d_tensor(test_inputs_raw, device=device)

    original_experts = [
        torch.load(f"{DATA_DIR}/experts/{layer}/expert_{i}.pt", weights_only=False).to(device).eval()
        for i in range(EXPERTS)
    ]
    gate = torch.load(f"{DATA_DIR}/experts/{layer}/gate.pt", weights_only=False).to(device).eval()
    shared_expert = torch.load(f"{DATA_DIR}/experts/{layer}/shared_expert.pt", weights_only=False).to(device).eval()
    shared_expert_gate = torch.load(f"{DATA_DIR}/experts/{layer}/shared_expert_gate.pt", weights_only=False).to(device).eval()

    group_modules = []
    built_groups = []
    r_by_group = {}
    layer_metrics = []
    for group_id, group in enumerate(groups):
        train_idx = selected_indices(train_acts, group, MAX_TRAIN_TOKENS_PER_GROUP)
        test_idx = selected_indices(test_acts, group, MAX_TEST_TOKENS_PER_GROUP)
        if len(train_idx) < MIN_TOKENS_PER_GROUP or len(test_idx) < MIN_TOKENS_PER_GROUP:
            continue
        train_x = train_all[torch.tensor(train_idx, device=device)]
        test_x = test_all[torch.tensor(test_idx, device=device)]
        experts = [original_experts[i] for i in group]
        with torch.no_grad():
            if method.endswith("_lszipit"):
                if len(group) != 2:
                    continue
                module = finetune(zipMLP)(experts[0], experts[1], train_x, create_new=True).to(device).eval()
                pair_metrics = test_model_on_data(module, experts[0], experts[1], test_x)
                vals = {
                    "MSE": pair_metrics["MSE"],
                    "normalized_MSE": pair_metrics["RelMSE"],
                    "Cosine": pair_metrics["Cosine"],
                    "CKA": pair_metrics["CKA"],
                    "JS": pair_metrics["JS"],
                }
                r_eff = experts[0].hid_features
            else:
                h_train = torch.cat([hidden_activations(expert, train_x) for expert in experts], dim=-1)
                y_train = torch.cat([expert(train_x) for expert in experts], dim=-1)
                y_test = torch.cat([expert(test_x) for expert in experts], dim=-1)
                d_ff = experts[0].hid_features
                r_req = max(1, int(round(cfg["alpha"] * d_ff)))
                if cfg["core"] == "mean":
                    h = h_train.float().reshape(h_train.shape[0], len(group), d_ff).mean(dim=1)
                    r_eff = min(r_req, h.shape[1])
                    h = h[:, :r_eff]
                    basis = None
                    mean = None
                else:
                    h, basis, mean, r_eff = build_pca_core(h_train, r_req)
                w = solve_ridge(h, y_train, cfg["lambda"])
                module = HypergraphGroup(
                    experts,
                    core=cfg["core"],
                    basis=basis.to(device) if basis is not None else None,
                    mean=mean.to(device) if mean is not None else None,
                    w=w.to(device),
                    group=group,
                ).to(device).eval()
                pred = torch.cat([module.forward_side(test_x, side) for side in range(len(group))], dim=-1)
                vals = reconstruction_metrics(y_test, pred)
        built_groups.append(group)
        group_modules.append(module)
        r_by_group[len(built_groups) - 1] = r_eff
        layer_metrics.append(vals)

    if not built_groups:
        raise RuntimeError(f"no fused groups for {method} layer {layer}")

    d_model = train_all.shape[-1]
    d_ff = original_experts[0].hid_features
    for group_id, group in enumerate(built_groups):
        if group_id not in r_by_group:
            r_by_group[group_id] = d_ff if method.endswith("_lszipit") else max(1, int(round(cfg["alpha"] * d_ff)))
    compression = layer_parameter_reduction(
        built_groups,
        num_experts=EXPERTS,
        d_model=d_model,
        d_ff=d_ff,
        r_by_group=r_by_group,
    )
    aggregate = {}
    if layer_metrics:
        for key in ["MSE", "normalized_MSE", "Cosine", "CKA", "JS"]:
            aggregate[key] = float(sum(m[key] for m in layer_metrics) / len(layer_metrics))
    runtime_layer = RuntimeFusionLayer(
        num_experts=EXPERTS,
        groups=built_groups,
        group_modules=group_modules,
        original_experts=original_experts,
        gate=gate,
        shared_expert=shared_expert,
        shared_expert_gate=shared_expert_gate,
    ).to(device).eval()
    meta = {**cfg, **compression, **aggregate}
    return runtime_layer, meta


def _join_or(items):
    if len(items) <= 1:
        return "".join(items)
    if len(items) == 2:
        return f"{items[0]} or {items[1]}"
    return ", ".join(items[:-1]) + f", or {items[-1]}"


def _extract_question(sample):
    if "question" in sample:
        return sample["question"]
    if "sentence" in sample:
        return sample["sentence"]
    if "question_stem" in sample:
        return sample["question_stem"]
    raise KeyError("unsupported sample format")


def _extract_choices(sample):
    if "choices" in sample:
        choices = sample["choices"]
        if isinstance(choices, dict):
            return list(choices["label"]), list(choices["text"])
        labels = [chr(ord("A") + i) for i in range(len(choices))]
        return labels, list(choices)
    if "option1" in sample and "option2" in sample:
        return ["A", "B"], [sample["option1"], sample["option2"]]
    raise KeyError("unsupported choices format")


def _extract_answer(sample, labels):
    if "answerKey" in sample:
        return sample["answerKey"]
    ans = sample["answer"]
    if isinstance(ans, int):
        return labels[ans]
    if ans in labels:
        return ans
    return labels[int(ans) - 1]


def _format_prompt(sample, tokenizer, letters):
    labels, texts = _extract_choices(sample)
    options = "".join(f"{label}. {text}\n" for label, text in zip(labels, texts))
    system = (
        "You are a precise evaluation system. "
        "You will be given a question and multiple-choice options. "
        f"Your task is to answer with ONLY the letter corresponding to the correct option ({_join_or(letters)}). "
        "Do not write the full answer, do not write explanations. Output only one character."
    )
    user = f"Question: {_extract_question(sample)}\nOptions:\n{options}\nAnswer:"
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def evaluate_qa(model, tokenizer, dataset_kwargs, n, *, seed=42, batch_size=8, max_length=512):
    set_seed(seed)
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"
    full = load_dataset(**dataset_kwargs)
    n = min(n, len(full))
    indices = sorted(random.Random(seed).sample(range(len(full)), n))
    subset = full.select(indices)
    answer_letters, _ = _extract_choices(subset[0])
    letter_to_idx = {letter: i for i, letter in enumerate(answer_letters)}
    answer_token_ids = [tokenizer.encode(letter, add_special_tokens=False)[0] for letter in answer_letters]
    answer_token_tensor = torch.tensor(answer_token_ids, device=model.device)
    prompts = [_format_prompt(s, tokenizer, answer_letters) for s in subset]
    ground_truths = [_extract_answer(s, answer_letters) for s in subset]
    correct = 0
    total = 0
    model.eval()
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i : i + batch_size]
        batch_truths = ground_truths[i : i + batch_size]
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(model.device)
        outputs = model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask, use_cache=False)
        last_logits = outputs.logits[:, -1, :]
        idx = answer_token_tensor.unsqueeze(0).expand(last_logits.size(0), -1)
        answer_logits = torch.gather(last_logits, 1, idx)
        predicted = torch.argmax(answer_logits, dim=1)
        truth = torch.tensor([letter_to_idx[t] for t in batch_truths], device=model.device)
        correct += int((predicted == truth).sum())
        total += len(batch_prompts)
        del inputs, outputs, last_logits, idx, answer_logits, predicted, truth
        torch.cuda.empty_cache()
    return correct / total

print("Hypergraph runtime helpers loaded.")


## Этап 1–5. Сбор артефактов


In [ ]:

# Stage 1-5: collect co-activation matrices, pairs, experts, MLP inputs, routing indices.

import gc

def collect_artifacts():
    set_seed(SEED)
    os.makedirs(DATA_DIR, exist_ok=True)
    print("Loading model for artifact collection...")
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device, torch_dtype=MODEL_DTYPE)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    max_layers = max(LAYERS) + 1
    while len(model.model.layers) > max_layers:
        del model.model.layers[-1]
    model = model.to(device)

    calib_train = load_dataset(**CALIB_TRAIN_DATASET)["question"][:TRAIN_SIZE]
    calib_val = load_dataset(**CALIB_VAL_DATASET)["question"][:VALIDATION_SIZE]

    class ExpertsLogger(nn.Module):
        def __init__(self, gate):
            super().__init__()
            self.gate = gate
            self.num_experts = gate.num_experts
            self.experts = torch.zeros((self.num_experts, self.num_experts), device="cpu")

        def forward(self, x):
            router_logits = self.gate(x)
            if isinstance(router_logits, tuple):
                selected_experts = router_logits[-1]
            else:
                routing_weights = router_logits.softmax(dim=1, dtype=torch.float)
                _, selected_experts = torch.topk(routing_weights, TOP_K, dim=-1)
            for experts_stack in selected_experts:
                for expert1 in experts_stack:
                    for expert2 in experts_stack:
                        self.experts[expert1, expert2] += 1
            return router_logits

        def clear(self):
            self.experts.zero_()

    loggers = {}
    for layer in LAYERS:
        gate = model.model.layers[layer].mlp.gate
        loggers[layer] = ExpertsLogger(gate.gate if hasattr(gate, "gate") else gate)
        model.model.layers[layer].mlp.gate = loggers[layer]

    def collect_coactivation(texts, split_name):
        with torch.no_grad():
            for logger in loggers.values():
                logger.clear()
            for text in tqdm(texts, desc=f"coact-{split_name}"):
                input_ids = tokenizer(text, return_tensors="pt")["input_ids"].to(device)
                model(input_ids)
                del input_ids
            act_dir = f"{DATA_DIR}/activations"
            os.makedirs(act_dir, exist_ok=True)
            for layer, logger in loggers.items():
                torch.save(logger.experts.clone(), f"{act_dir}/{split_name}_matrix_{layer}.pt")

    collect_coactivation(calib_train, "train")
    collect_coactivation(calib_val, "test")

    os.makedirs(f"{DATA_DIR}/pairs", exist_ok=True)
    for layer in LAYERS:
        _ = compute_activation_correlation_matrix(layer, DATA_DIR)
        pairs = get_pairs(
            method=PAIRING_METHOD,
            layer=layer,
            n_experts=EXPERTS,
            data_dir=DATA_DIR,
            device=device,
            evolution_steps=EVOLUTION_STEPS,
            progress=True,
        )
        torch.save(pairs, f"{DATA_DIR}/pairs/{layer}.pt")

    print("Saving expert weights on CPU...")
    model_cpu = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=MODEL_DTYPE, device_map="cpu")
    while len(model_cpu.model.layers) > max_layers:
        del model_cpu.model.layers[-1]
    for layer in tqdm(LAYERS, desc="experts"):
        experts = model_cpu.model.layers[layer].mlp.experts
        os.makedirs(f"{DATA_DIR}/experts/{layer}", exist_ok=True)
        for i, expert in enumerate(experts):
            torch.save(
                turn_in_mlp(expert, device=torch.device("cpu")).to(dtype=model_cpu.dtype),
                f"{DATA_DIR}/experts/{layer}/expert_{i}.pt",
            )
        torch.save(model_cpu.model.layers[layer].mlp.gate, f"{DATA_DIR}/experts/{layer}/gate.pt")
        torch.save(model_cpu.model.layers[layer].mlp.shared_expert, f"{DATA_DIR}/experts/{layer}/shared_expert.pt")
        torch.save(model_cpu.model.layers[layer].mlp.shared_expert_gate, f"{DATA_DIR}/experts/{layer}/shared_expert_gate.pt")
    del model_cpu
    gc.collect()

    class MLPInputLogger(nn.Module):
        def __init__(self, module):
            super().__init__()
            self.module = module
            self.data = []

        def forward(self, *args, **kwargs):
            hidden = args[0]
            for token in hidden.detach().cpu()[0]:
                self.data.append(token)
            return self.module(*args, **kwargs)

    for layer in LAYERS:
        logger = MLPInputLogger(model.model.layers[layer].mlp)
        model.model.layers[layer].mlp = logger
        os.makedirs(f"{DATA_DIR}/mlp_inputs", exist_ok=True)
        for split_name, texts in [("train", calib_train), ("test", calib_val)]:
            logger.data = []
            for text in tqdm(texts, desc=f"mlp-{split_name}-L{layer}"):
                input_ids = tokenizer(text, return_tensors="pt")["input_ids"].to(device)
                model(input_ids)
            torch.save(logger.data, f"{DATA_DIR}/mlp_inputs/{split_name}_{layer}.pt")
        model.model.layers[layer].mlp = logger.module

    for layer in LAYERS:
        train_inputs = torch.load(f"{DATA_DIR}/mlp_inputs/train_{layer}.pt", weights_only=False)
        test_inputs = torch.load(f"{DATA_DIR}/mlp_inputs/test_{layer}.pt", weights_only=False)

        def route_indices(inputs):
            out = []
            for example in tqdm(inputs, desc=f"route-L{layer}"):
                router_output = model.model.layers[layer].mlp.gate(example.to(device))
                selected = router_output[-1] if isinstance(router_output, tuple) else router_output.topk(TOP_K, dim=-1)[1]
                out.append(selected.cpu().flatten())
            return out

        os.makedirs(f"{DATA_DIR}/activations", exist_ok=True)
        torch.save(route_indices(train_inputs), f"{DATA_DIR}/activations/train_all_{layer}.pt")
        torch.save(route_indices(test_inputs), f"{DATA_DIR}/activations/test_all_{layer}.pt")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("Artifact collection complete:", DATA_DIR)

marker = Path(DATA_DIR) / "activations" / f"train_matrix_{LAYERS[0]}.pt"
if SKIP_ARTIFACTS_IF_PRESENT and marker.exists():
    print("Artifacts already present, skipping collection:", DATA_DIR)
else:
    collect_artifacts()


## Часть A. Single-layer sweep (все методы × все слои)


In [ ]:

# Part A: single-layer downstream proxy for every method and layer.

set_seed(SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
single_csv = Path(RESULTS_DIR) / "single_layer_replacement.csv"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device, torch_dtype=MODEL_DTYPE)
max_layers = max(LAYERS) + 1
while len(model.model.layers) > max_layers:
    del model.model.layers[-1]

base_accuracy = evaluate_qa(model, tokenizer, QA_TASK, QA_NUM_EXAMPLES, seed=SEED, batch_size=QA_BATCH_SIZE)
print(f"Base accuracy ({QA_TASK}, n={QA_NUM_EXAMPLES}): {base_accuracy:.4f}")

rows = []
cases = [(method, layer) for method in COMPARISON_METHODS for layer in LAYERS]
for case_idx, (method, layer) in enumerate(tqdm(cases, desc="single-layer"), start=1):
    old_mlp = model.model.layers[layer].mlp
    try:
        fused_layer, meta = build_runtime_layer(layer, method, device)
        model.model.layers[layer].mlp = fused_layer
        acc = evaluate_qa(model, tokenizer, QA_TASK, QA_NUM_EXAMPLES, seed=SEED, batch_size=QA_BATCH_SIZE)
        row = {
            "method": method,
            "layer": layer,
            "core": meta["core"],
            "alpha": meta["alpha"],
            "lambda": meta["lambda"],
            "CR_layer_actual": meta["CR_actual"],
            "m2": meta["num_m2"],
            "m3": meta["num_m3"],
            "m4": meta["num_m4"],
            "overlap": meta["overlap_count"],
            "base_accuracy": base_accuracy,
            "single_layer_accuracy": acc,
            "delta_accuracy": base_accuracy - acc,
            "MSE": meta["MSE"],
            "nMSE": meta["normalized_MSE"],
            "Cosine": meta["Cosine"],
            "CKA": meta["CKA"],
            "JS": meta["JS"],
            "proxy_task": str(QA_TASK),
        }
        rows.append(row)
    except Exception as exc:
        print(f"FAILED method={method} layer={layer}: {exc}")
    finally:
        model.model.layers[layer].mlp = old_mlp
        torch.cuda.empty_cache()

if rows:
    fields = list(rows[0].keys())
    with single_csv.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
print("Saved:", single_csv)


## Часть B. Cumulative lossless compression


In [ ]:

# Part B: method-specific best-first schedules and cumulative lossless compression.

import pandas as pd

single_csv = Path(RESULTS_DIR) / "single_layer_replacement.csv"
df = pd.read_csv(single_csv)
schedule_dir = Path(RESULTS_DIR) / "schedules"
schedule_dir.mkdir(parents=True, exist_ok=True)

def write_schedule(method):
    part = df[df["method"] == method].sort_values(["layer", "delta_accuracy"]).groupby("layer", as_index=False).first()
    part = part.sort_values("delta_accuracy", ascending=True)
    payload = {
        "method": method,
        "proxy_task": str(QA_TASK),
        "schedule_type": "method_specific",
        "schedule": [
            {
                "rank": rank,
                "layer": int(row["layer"]),
                "delta_accuracy": float(row["delta_accuracy"]),
                "CR_layer_actual": float(row["CR_layer_actual"]),
                "MSE": float(row["MSE"]),
                "group_structure": f"{int(row['m2'])}/{int(row['m3'])}/{int(row['m4'])}",
            }
            for rank, row in enumerate(part.to_dict("records"))
        ],
    }
    path = schedule_dir / "method_specific" / f"{method}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return payload

schedules = {method: write_schedule(method) for method in COMPARISON_METHODS}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device, torch_dtype=MODEL_DTYPE)
max_layers = max(LAYERS) + 1
while len(base_model.model.layers) > max_layers:
    del base_model.model.layers[-1]
base_accuracy = evaluate_qa(base_model, tokenizer, QA_TASK, QA_NUM_EXAMPLES, seed=SEED, batch_size=QA_BATCH_SIZE)
del base_model
torch.cuda.empty_cache()

cumulative_csv = Path(RESULTS_DIR) / "lossless_cumulative_results.csv"
summary_rows = []

for method in COMPARISON_METHODS:
    schedule = schedules[method]["schedule"]
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map=device, torch_dtype=MODEL_DTYPE)
    while len(model.model.layers) > max_layers:
        del model.model.layers[-1]
    actual_cr_sum = 0.0
    compressed_layers = []
    best_lossless = None
    for step_idx, item in enumerate(schedule, start=1):
        layer = int(item["layer"])
        fused_layer, meta = build_runtime_layer(layer, method, device)
        model.model.layers[layer].mlp = fused_layer
        compressed_layers.append(layer)
        actual_cr_sum += float(meta["CR_actual"]) / len(model.model.layers)
        acc = evaluate_qa(model, tokenizer, QA_TASK, QA_NUM_EXAMPLES, seed=SEED, batch_size=QA_BATCH_SIZE)
        delta = base_accuracy - acc
        row = {
            "method": method,
            "schedule_type": "method_specific",
            "step": step_idx,
            "actual_CR": actual_cr_sum,
            "compressed_layers": len(compressed_layers),
            "compressed_layer_list": json.dumps(compressed_layers),
            "accuracy": acc,
            "base_accuracy": base_accuracy,
            "delta_accuracy": delta,
            "near_lossless": delta <= LOSSLESS_THRESHOLD,
            "total_m2": meta["num_m2"],
            "total_m3": meta["num_m3"],
            "total_m4": meta["num_m4"],
        }
        fields = list(row.keys())
        exists = cumulative_csv.exists()
        with cumulative_csv.open("a", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            if not exists:
                writer.writeheader()
            writer.writerow(row)
        if row["near_lossless"]:
            best_lossless = row
        else:
            break
    if best_lossless is None:
        best_lossless = {
            "method": method,
            "schedule_type": "method_specific",
            "step": 0,
            "actual_CR": 0.0,
            "compressed_layers": 0,
            "accuracy": base_accuracy,
            "base_accuracy": base_accuracy,
            "delta_accuracy": 0.0,
            "near_lossless": True,
        }
    summary_rows.append(best_lossless)
    del model
    torch.cuda.empty_cache()

summary_csv = Path(RESULTS_DIR) / "max_lossless_summary.csv"
if summary_rows:
    fields = sorted({key for row in summary_rows for key in row.keys()})
    with summary_csv.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(summary_rows)
print("Saved:", cumulative_csv)
print("Saved:", summary_csv)


## Итоговая таблица и график


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

summary = pd.read_csv(Path(RESULTS_DIR) / "max_lossless_summary.csv")
summary = summary.sort_values("actual_CR", ascending=False)
display(summary[["method", "actual_CR", "compressed_layers", "accuracy", "delta_accuracy"]])

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(summary["method"], summary["actual_CR"], color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
ax.axhline(LOSSLESS_THRESHOLD, color="black", linestyle="--", linewidth=1, label="lossless threshold (delta)")
ax.set_ylabel("Max lossless routed-expert CR")
ax.set_title(f"Method comparison on {QA_TASK['path']} (n={QA_NUM_EXAMPLES})")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig(Path(RESULTS_DIR) / "max_lossless_comparison.png", dpi=150)
plt.show()
print("Representative comparison table saved to", RESULTS_DIR)
